# Expectation-Maximization (gentle)

_Machine Learning_

**Soft clustering when each point partly belongs to several groups.**

k-means gives each point to exactly one cluster. EM (Expectation–Maximization) lets a point partly belong to several.

     Each point gets a responsibility: the chance it came from each cluster.

---

This notebook builds up Expectation–Maximization one piece at a time: first a Gaussian mixture on clean blobs to see the **soft responsibilities**, then the same idea on real wine data. Run each cell and read the note above it to follow how each point gets shared across clusters. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Fit a Gaussian mixture to three blobs

We make 400 points drawn from three well-separated clusters. A `GaussianMixture` runs the **EM algorithm** under the hood: it alternates between estimating each point's cluster responsibilities and re-fitting the clusters, until the fit stops changing. `converged_` tells us EM reached a stable answer.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

X, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.0, random_state=0)

gmm = GaussianMixture(n_components=3, random_state=0).fit(X)
print("converged:", gmm.converged_)
print("mixture weights:", np.round(gmm.weights_, 3))

### Step 2 — Read the soft memberships

This is what makes EM different from k-means. Instead of one hard label per point, `predict_proba` returns a **responsibility** for every cluster: `P(component | point)`. Each row sums to 1, so a point can be, say, 80% cluster A and 20% cluster B rather than forced entirely into one.

In [ ]:
# soft assignments: each row is P(component | point), sums to 1
probs = gmm.predict_proba(X[:3])
print("soft memberships of first 3 points:")
print(np.round(probs, 3))

## Visualize the data & results

_Can a Gaussian mixture softly assign each real wine to one of three overlapping cultivar groups?_

### Step 1 — Load real wine data and reduce it to 2D

Real wines have 13 chemical measurements on very different scales, so we **standardize** them (zero mean, unit variance) so no single feature dominates. We then use **PCA** to project the 13 dimensions down to 2, purely so we can plot the clusters on a flat chart.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

wine = load_wine()
X = StandardScaler().fit_transform(wine.data)        # put all features on the same scale
P = PCA(n_components=2, random_state=0).fit_transform(X)   # squash to 2D for plotting

### Step 2 — Fit the mixture and pick each point's top cluster

We fit a 3-component mixture on the 2D wine points. EM again produces soft responsibilities, but to color the plot we take a **hard label** for each point — the cluster with the highest responsibility (the `argmax` of its soft probabilities).

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=0).fit(P)
labels = gmm.predict(P)            # hard label = argmax of soft probs
print("mixture weights", gmm.weights_)

### Step 3 — Plot the soft clusters

We draw each cluster in its own color. Where the three groups overlap, the soft responsibilities were genuinely split — those border wines partly belong to more than one cultivar group, which is exactly the flexibility EM buys us over hard clustering.

In [ ]:
colors = ["#4ea1ff", "#7ee787", "#c89bff"]
for c in range(3):
    pts = P[labels == c]
    plt.scatter(pts[:, 0], pts[:, 1], c=colors[c], edgecolor="k",
                label="component %d" % (c + 1))
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("GMM soft clusters on wine")
plt.legend()
plt.show()